# Spark UI Walkthrough

## Your Window into the Engine

Spark ships with a built-in web interface that gives you real-time and historical visibility into every computation your application performs. Learning to read it is **the single most important debugging skill** for a Spark developer — it answers questions like:

- Why is this query slow?
- Which stage is the bottleneck?
- Is my data skewed?
- Is caching actually helping?
- Are my executors being used evenly?

## Two UIs to Know

| URL | What it shows |
|-----|---------------|
| `http://localhost:8080` | **Spark Master UI** — lists all submitted applications (running and completed), total cluster resources (CPU cores, RAM), and per-worker status. |
| `http://localhost:4040` | **Application UI** — per-application detail: Jobs, Stages, Tasks, Storage, Executors, SQL, Environment. Only available while the application is running. |

Open **both tabs now** in your browser before running any cells. You will watch them update in real time as you execute the code below.

> **Tip:** If port 4040 is already taken (by a previous session), Spark tries 4041, 4042, … Check the SparkContext log output for the actual port.

## Spark Application UI — Tab Reference

| Tab | What to look for |
|-----|------------------|
| **Jobs** | Every Action (`.count()`, `.show()`, `.write`) creates a Job. Green = succeeded, red = failed. Click a job to see its stages. |
| **Stages** | A Job is split into Stages wherever data must be shuffled across the network. Each stage shows input/output bytes, shuffle read/write, and duration. A slow stage with large shuffle write is your bottleneck. |
| **Tasks** | Each Stage runs one Task per partition. Look at the task duration histogram — a very long tail means data skew (one partition is much larger than the rest). The event timeline shows whether tasks ran in parallel or serially. |
| **Storage** | Shows DataFrames and RDDs that have been cached/persisted. Columns: Memory Used, Disk Used, Cached Partitions. If Memory Used is 0 after a `.cache()`, the cache was evicted — increase executor memory. |
| **Executors** | Lists every executor JVM. Columns: cores, memory used, GC time, shuffle read/write, task time. High GC time (> 10% of task time) means your executor heap is too small. |
| **SQL / DataFrame** | Shows the Catalyst logical plan, optimised logical plan, and physical plan for every SQL query and DataFrame Action. The DAG visualiser links plan nodes to timing data. **This is the most useful tab for query tuning.** |
| **Environment** | Dumps every Spark configuration property, JVM flags, and classpath entry. Useful for confirming that your `.config()` calls actually took effect. |

In [14]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("06 - Spark UI Walkthrough")
    # Keep partition count small so the UI is easier to read
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

# The URL printed here is where the Application UI lives.
# It maps to http://localhost:4040 if you are running with Docker port forwarding.
print(f"Spark {spark.version} ready.")
print(f"Application UI : {spark.sparkContext.uiWebUrl}")

Spark 3.5.3 ready.
Application UI : http://jupyter:4040


In [15]:
# ── Multi-stage job: watch the Jobs and Stages tabs ──────────────────────────
#
# A "stage boundary" occurs whenever Spark must SHUFFLE data — i.e., move rows
# from one executor to another so that the same grouping key ends up on the
# same machine. This is required by groupBy, join, distinct, repartition, etc.
#
# This cell deliberately triggers at least TWO stages:
#   Stage 1 (map / partial aggregation): read the range, compute partial sums
#   Stage 2 (reduce / final aggregation): shuffle partial sums by 'group_key',
#                                         then compute final totals
#
# => While this cell runs, go to localhost:4040 -> Jobs -> click the running job
#    -> you will see the two-stage DAG with a shuffle boundary between them.

# Generate a synthetic dataset: 1 million rows, 10 group keys
large_df = (
    spark.range(0, 1_000_000)            # creates a single-column DataFrame 'id'
    .withColumn("group_key", (F.col("id") % 10).cast("string"))   # 10 groups
    .withColumn("value", (F.rand() * 100).cast("double"))          # random metric
)


# groupBy causes a shuffle — this is the stage boundary
result_df = (
    large_df
    .groupBy("group_key")
    .agg(
        F.count("*").alias("row_count"),
        F.round(F.avg("value"), 4).alias("avg_value"),
        F.round(F.sum("value"), 2).alias("total_value"),
    )
    .orderBy("group_key")
)

# Write to disk to create a concrete Action (triggers the full plan)
OUTPUT_PATH = "/home/jovyan/data/ui_demo_result.parquet"
result_df.write.mode("overwrite").parquet(OUTPUT_PATH)

# Also show the result
spark.read.parquet(OUTPUT_PATH).show()
print("\nNow check localhost:4040 -> Jobs to see the completed job and its stages.")

+---------+---------+---------+-----------+
|group_key|row_count|avg_value|total_value|
+---------+---------+---------+-----------+
|        0|   100000|  50.0068| 5000684.41|
|        1|   100000|  49.8244| 4982444.99|
|        2|   100000|  50.0504|  5005036.0|
|        3|   100000|   49.967| 4996700.14|
|        4|   100000|  49.9474|  4994744.0|
|        5|   100000|  50.0335| 5003346.48|
|        6|   100000|  50.0164|  5001643.1|
|        7|   100000|  50.0648| 5006475.82|
|        8|   100000|  50.0179| 5001791.16|
|        9|   100000|  50.0699| 5006986.38|
+---------+---------+---------+-----------+


Now check localhost:4040 -> Jobs to see the completed job and its stages.


## Stage Boundaries: Why Shuffles Split Your Job

Spark executes transformations **lazily** — it builds up a DAG (Directed Acyclic Graph) of operations and only runs them when an Action is called. Within that DAG, Spark divides the work into **Stages**.

A new stage begins whenever the computation requires a **shuffle** — a network-wide redistribution of data. Here is why:

```
  Executor 1            Executor 2
  ──────────            ──────────
  [id=0..249k]          [id=250k..499k]
       │                      │
  partial group_key=0       partial group_key=0  <─── both executors have
  partial group_key=1       partial group_key=1       data for key 0 !
       │                      │
       └─── SHUFFLE ──────────┘
              │
         Executor 1 now holds ALL rows for group_key=0,1
         Executor 2 now holds ALL rows for group_key=2,3  …
              │
         Final aggregation (Stage 2)
```

**Shuffles are expensive** because they:
1. Serialise data to disk (shuffle write)
2. Transfer it over the network
3. Deserialise and sort it on the receiving side (shuffle read)

In the Stages tab you will see **Shuffle Read** and **Shuffle Write** columns. These numbers tell you how much data crossed the network. Minimising shuffle volume is the primary goal of query optimisation.

**Transformations that cause a shuffle:** `groupBy`, `join`, `distinct`, `repartition`, `sort`/`orderBy`  
**Transformations that do NOT cause a shuffle:** `filter`, `select`, `withColumn`, `limit`, `map`, `flatMap`

In [16]:
# ── Caching: watch the Storage tab fill up ────────────────────────────────────
#
# df.cache() marks the DataFrame for in-memory caching.
# The cache is NOT populated until the first Action that touches the data.
# Subsequent Actions on the same cached DataFrame skip recomputation entirely.
#
# => After the FIRST count() below: go to Storage tab — you should see
#    'large_df' appear with non-zero Memory Used.
# => Run the SECOND count() and note it completes in milliseconds instead
#    of seconds — the data is served directly from executor memory.

import time

# Cache the large DataFrame so it survives across multiple Actions
large_df.cache()   # lazy — nothing happens yet
print("cache() called — the data is NOT in memory yet (lazy operation)")


cache() called — the data is NOT in memory yet (lazy operation)


In [17]:

# First count(): triggers the full read + cache population
t0 = time.time()
c1 = large_df.count()
t1 = time.time()
print(f"First  count() = {c1:,}   took {t1 - t0:.2f}s  (reads from source + fills cache)")
print("  => Check Storage tab now — large_df should appear with Memory Used > 0")

# Second count(): served entirely from executor memory — much faster
t2 = time.time()
c2 = large_df.count()
t3 = time.time()
print(f"Second count() = {c2:,}   took {t3 - t2:.2f}s  (served from in-memory cache)")

speedup = (t1 - t0) / max(t3 - t2, 0.001)
print(f"\nSpeedup from caching: ~{speedup:.0f}x")
print("  => In the Jobs tab, the second count() job shows many 'skipped' stages (green).")

# Unpersist when you no longer need the cache to free executor memory
#large_df.unpersist()
#print("\nunpersist() called — memory released back to the executor pool.")

First  count() = 1,000,000   took 0.88s  (reads from source + fills cache)
  => Check Storage tab now — large_df should appear with Memory Used > 0
Second count() = 1,000,000   took 0.13s  (served from in-memory cache)

Speedup from caching: ~7x
  => In the Jobs tab, the second count() job shows many 'skipped' stages (green).


## Reading the DAG Visualiser

In the **SQL / DataFrame** tab, click on any query to open the DAG (Directed Acyclic Graph) visualiser. Each box is a plan node (an operator); arrows show the data flow from bottom (source) to top (sink).

### Colour coding

| Colour | Meaning |
|--------|---------|
| **Blue** (solid border) | This node performed a shuffle — data crossed the network between executors. This is a stage boundary. |
| **Green** (with a dot) | This RDD or DataFrame partition was served from the in-memory cache. The stage was **skipped** entirely — no computation ran. |
| **Gray / white** | Regular computation — no shuffle, no cache hit. |

### What to look for when tuning

1. **Blue nodes with large shuffle write numbers** — these are your bottleneck. Consider:  
   - Reducing the number of distinct keys (fewer partitions needed after shuffle)  
   - Using `broadcast join` to avoid a sort-merge join shuffle entirely  
   - Lowering `spark.sql.shuffle.partitions` if 200 (the default) is too many for your data size

2. **Stages with many small tasks vs one giant task** — this is data skew. See the next cell for a demonstration.

3. **Green (cached) nodes** — if a node you expected to be green is blue or gray, the cache was evicted. Add more executor memory or choose `DISK_AND_MEMORY` persistence level.

4. **Very deep DAGs (many nodes)** — long chains of `.withColumn()` calls can create huge plans. Use `df.checkpoint()` at natural break points to truncate the lineage.

In [19]:
# ── Task skew demonstration: repartition(1) forces all work onto one task ─────
#
# Normally Spark distributes partitions evenly across tasks.
# repartition(1) collapses the entire DataFrame into a SINGLE partition,
# which means a SINGLE task must process all the data.
#
# In the Tasks tab you will see:
#   - 1 task that runs for ~N seconds
#   - All other "tasks" are instantaneous (they have nothing to do)
#
# This is the extreme end of DATA SKEW — in real life skew is subtler but
# the pattern in the duration histogram looks the same: one bar far to the
# right while most bars cluster near zero.
#
# => While this cell runs, go to:
#    localhost:4040 -> Jobs -> click the running job -> click the shuffle stage
#    -> Tasks tab: sort by Duration descending
#    You will see one task doing all the work, and the rest finishing instantly.

print("Starting skew demonstration — open localhost:4040 -> Jobs NOW...")

# Start with a reasonably partitioned DataFrame (default parallelism)
skew_df = spark.range(0, 500_000).withColumn("value", F.rand() * 100)

# repartition(1) — ALL data moves to a single partition = a single task
# This triggers a full shuffle just to collapse everything to one partition.
skewed = (
    skew_df
    .repartition(1)                             # <-- the skew culprit
    .withColumn("value_squared", F.col("value") ** 2)
    .withColumn("value_log",     F.log(F.col("value") + 1))
)

# Action: count triggers the full execution
total = skewed.count()
print(f"\nRow count: {total:,}")
print("\nIn the Tasks tab you should have seen:")
print("  - Stage with repartition: 1 task did all the work")
print("  - All other tasks: near-zero duration (nothing to process)")
print("\nFix: avoid repartition(1). Use .coalesce(1) only at the final write step")
print("     if you NEED a single output file. Otherwise let Spark choose parallelism.")

Starting skew demonstration — open localhost:4040 -> Jobs NOW...

Row count: 500,000

In the Tasks tab you should have seen:
  - Stage with repartition: 1 task did all the work
  - All other tasks: near-zero duration (nothing to process)

Fix: avoid repartition(1). Use .coalesce(1) only at the final write step
     if you NEED a single output file. Otherwise let Spark choose parallelism.


In [20]:
# Release all cluster resources.
# After this call, localhost:4040 will no longer be available
# (the application UI is torn down with the SparkContext).
# The completed application will still appear in the Master UI at localhost:8080.
spark.stop()
print("SparkSession stopped.")
print("The application is now visible under 'Completed Applications' at localhost:8080.")

SparkSession stopped.
The application is now visible under 'Completed Applications' at localhost:8080.
